### Ingest Auction Results CSV
This notebook ingests the auction item results file from the landing layer, standardizes the dataset, and writes it to a bronze Delta table.

Business context:
* This table shows the auction results at the lot or item level
* Each row represents a catalog item with auction metadata, pricing, status, and descriptive attributes
* The bronze table preserves the raw auction results so they can support downstream reporting, settlement analysis, and catalog performance tracking

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the auction results file path
This step assembles the full storage path for the raw auction item results CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific dated auction results file to ingest
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the auction results schema
This step declares the expected structure of the CSV so Spark reads item, auction, pricing, and attribute fields with the intended data types.

What happens here:
* Defines the incoming auction results columns explicitly
* Applies consistent types for estimates, hammer price, realized price, and counters
* Avoids schema drift that can happen with CSV inference alone


#### Auction results data dictionary
This file represents auction results at the item or lot level, where each record describes an auction lot together with pricing, status, catalog details, and product attributes.

Key fields:
* `AuctionName` and `AuctionDate`: identify the auction event where the item appeared
* `ItemNum`, `ReferenceNumber`, `LotNumber`, and `LotNumberExtension`: item and lot identifiers used across cataloging and auction operations
* `LotName*` and `LotDescription`: descriptive details about the auction lot
* `Category` and `Attribute:*` fields: item characteristics such as format, country, region, variety, type, and item_year
* `InventoryStatus` and `ShippingStatus`: operational status fields for the lot after cataloging or sale
* `StartingBid`, `LowEstimate`, `HighEstimate`, `HammerPrice`, `CommissionAmount`, `BuyersPremium`, and `PriceRealized`: commercial fields used to evaluate lot pricing and auction outcomes
* `Consignor #`, `ConsignorName`, and related cataloger fields: ownership and internal workflow information for the lot
* `FavoritesCount`: a measure of bidder or viewer interest in the item

Business use:
* Analyze auction performance at the lot level
* Track estimates, realized values, and settlement-related metrics
* Support catalog, consignor, and post-sale operational reporting


#### Step 4: Load and enrich the auction results data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the auction item results file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready results dataset
This step standardizes the DataFrame so the columns are easier to use after the bronze write.

What happens here:
* Renames source columns into a lowercase, underscore-based format
* Cleans spaces and special characters in field names for downstream usability
* Confirms the bronze target context before persisting the table


#### Step 6: Write the bronze auction results table
This final step persists the cleaned auction item results DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed auction results data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze dataset available for downstream silver transformations, reporting, and settlement analysis


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/items_*.csv'

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Bronze-layer schema: everything as string, since typing/casting
# happens downstream in silver. This avoids inference issues and
# lets bronze absorb schema drift without breaking.
schema_auction_item = StructType([
    StructField('AuctionName', StringType(), True),
    StructField('AuctionDate', StringType(), True),
    StructField('ItemNum', StringType(), True),
    StructField('ReferenceNumber', StringType(), True),
    StructField('LotName*', StringType(), True),
    StructField('LotNumber', StringType(), True),
    StructField('LotNumberExtension', StringType(), True),
    StructField('Link', StringType(), True),
    StructField('Provenance', StringType(), True),
    StructField('Category', StringType(), True),
    StructField('ExportedImgs', StringType(), True),
    StructField('Consignor #', StringType(), True),
    StructField('ConsignorName', StringType(), True),
    StructField('ConsignorEmail', StringType(), True),
    StructField('Consignment #', StringType(), True),
    StructField('Consignment Manager', StringType(), True),
    StructField('CatalogerName', StringType(), True),
    StructField('CatalogerEmail', StringType(), True),
    StructField('Cataloged', StringType(), True),
    StructField('Photographed', StringType(), True),
    StructField('Approved', StringType(), True),
    StructField('InventoryStatus', StringType(), True),
    StructField('ShippingStatus', StringType(), True),
    StructField('TotalSettlement', StringType(), True),
    StructField('Attribute:Format', StringType(), True),
    StructField('Attribute:Country', StringType(), True),
    StructField('Attribute:Region', StringType(), True),
    StructField('Attribute:Sub Region', StringType(), True),
    StructField('Attribute:Locality', StringType(), True),
    StructField('Attribute:Variety', StringType(), True),
    StructField('Attribute:Type', StringType(), True),
    StructField('Attribute:Year', StringType(), True),
    StructField('Attribute:Category', StringType(), True),
    StructField('LotDescription', StringType(), True),
    StructField('Condition', StringType(), True),
    StructField('StartingBid', StringType(), True),
    StructField('Price', StringType(), True),
    StructField('LowEstimate', StringType(), True),
    StructField('HighEstimate', StringType(), True),
    StructField('Cost', StringType(), True),
    StructField('Reserveprice', StringType(), True),
    StructField('HammerPrice', StringType(), True),
    StructField('CommissionAmount', StringType(), True),
    StructField('BuyersPremium', StringType(), True),
    StructField('PriceRealized', StringType(), True),
    StructField('FavoritesCount', StringType(), True),
    StructField('TaxJarCategory', StringType(), True),
    StructField('Note', StringType(), True),
    StructField('DisableTax', StringType(), True),
    StructField('CustomFields', StringType(), True)
])

In [ ]:
raw_df = (
spark.read.format('csv') \
    .option('header', 'true') \
    .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
        .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .load(path)
)

target_schema = schema_auction_item

select_exprs = []
source_cols = set(raw_df.columns)

for field in target_schema.fields:
    if field.name in source_cols:
        select_exprs.append(F.col(field.name).cast(field.dataType).alias(field.name))
    else:
        select_exprs.append(F.lit(None).cast(field.dataType).alias(field.name))

final_df = raw_df.select(*select_exprs)
df_auction_item = add_columns(final_df)

In [ ]:
df_auction_item = df_auction_item.withColumnsRenamed({col: col.replace(' ', '_').lower() for col in df_auction_item.columns})

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_auction_item \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.auctionresults')